# 03 — Baseline dissolved-oxygen regression

**Experimental claim:** estimate dissolved oxygen from contemporaneously measured physicochemical river-water variables. This is not future forecasting, safety classification or ecological-status assessment.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocessing import (
    WATER_QUALITY_BASELINE_FEATURES,
    WATER_QUALITY_GROUP,
    WATER_QUALITY_TARGET,
    grouped_station_holdout,
)
from src.train import run_water_quality_baseline_experiment

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'water_quality_modeling.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
data = pd.read_csv(DATA_PATH)
print(f'Loaded processed modelling data: {data.shape}')

## Target and feature set

The six chemistry inputs are accompanied by below-/above-limit flags. Station, date and identifiers remain metadata and are excluded from the model matrix.

In [ ]:
print('Target:', WATER_QUALITY_TARGET)
print('Features:')
for feature in WATER_QUALITY_BASELINE_FEATURES:
    print('-', feature)
assert WATER_QUALITY_TARGET not in WATER_QUALITY_BASELINE_FEATURES
assert 'Q_DO' not in WATER_QUALITY_BASELINE_FEATURES

## Station-grouped holdout

The primary split holds out complete stations. This tests transfer to locations unseen during training and avoids same-station leakage.

In [ ]:
train_indices, test_indices = grouped_station_holdout(data, test_size=0.2, random_state=42)
train_stations = set(data.iloc[train_indices][WATER_QUALITY_GROUP])
test_stations = set(data.iloc[test_indices][WATER_QUALITY_GROUP])
overlap = train_stations & test_stations
assert not overlap
print('Train rows:', len(train_indices))
print('Test rows:', len(test_indices))
print('Train stations:', len(train_stations))
print('Test stations:', len(test_stations))
print('Station overlap:', len(overlap))

## Run or load the executed experiment

Set `RUN_EXPERIMENT = True` to refit all pipelines and regenerate artifacts. The committed results displayed below were generated by this same function with random state 42.

In [ ]:
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    experiment = run_water_quality_baseline_experiment(
        data, model_dir=MODEL_DIR, results_dir=RESULTS_DIR
    )
    print('Experiment rerun complete.')
else:
    print('Using saved results from the verified Phase 3 execution.')

## Station-grouped baseline comparison

In [ ]:
baseline_results = pd.read_csv(RESULTS_DIR / 'baseline_results.csv')
display(baseline_results[['model', 'mae', 'rmse', 'r2']].rename(columns={
    'model': 'Model', 'mae': 'MAE', 'rmse': 'RMSE', 'r2': 'R²'
}))
best = baseline_results.sort_values('mae').iloc[0]
print(f"Lowest grouped-holdout MAE: {best['model']} ({best['mae']:.3f} mg/L)")

## Temporal sensitivity

Training through 2018 and testing on 2019–2024 asks whether earlier learned relationships remain useful later. It does not make this a forecasting model.

In [ ]:
temporal_results = pd.read_csv(RESULTS_DIR / 'temporal_sensitivity_results.csv')
display(temporal_results[['model', 'mae', 'rmse', 'r2']].rename(columns={
    'model': 'Model', 'mae': 'MAE', 'rmse': 'RMSE', 'r2': 'R²'
}))

## Error analysis

The lowest and highest groups are defined by held-out target deciles. They are descriptive error ranges, not ecological classes or regulatory thresholds.

In [ ]:
range_errors = pd.read_csv(RESULTS_DIR / 'error_by_do_range.csv')
largest_errors = pd.read_csv(RESULTS_DIR / 'largest_absolute_errors.csv')
display(range_errors)
display(largest_errors.head(10))

In [ ]:
for filename in [
    'actual_vs_predicted.png',
    'residual_distribution.png',
    'residual_vs_predicted.png',
    'absolute_error_vs_actual.png',
]:
    display(Image(filename=str(RESULTS_DIR / 'figures' / filename)))

## Random Forest feature importance

Permutation importance reports the held-out MAE increase after shuffling one input. Impurity and permutation importance describe predictive association in this fitted model; neither establishes causation.

In [ ]:
impurity = pd.read_csv(RESULTS_DIR / 'random_forest_impurity_importance.csv')
permutation = pd.read_csv(RESULTS_DIR / 'random_forest_permutation_importance.csv')
print('Impurity importance:')
display(impurity.head(15))
print('Held-out permutation importance:')
display(permutation)

## Interpretation limits

The experiment shows that six co-measured chemistry variables and their qualifier flags contain some predictive information about dissolved oxygen at held-out stations. It does not prove causal effects, predict future oxygen, determine official ecological status, or assess drinking-water safety. Temperature is absent, censoring is handled only with transparent indicators, and extreme target values produce large errors.